* kolla "SamePortrait" om föra tillbaka det till Wikidata om trovärdigt
* 

In [1]:
from datetime import datetime
start_time  = datetime.now()
print("Last run: ", start_time)

Last run:  2026-03-12 18:25:33.878626


In [2]:
import requests
import concurrent.futures
from SPARQLWrapper import SPARQLWrapper, JSON
import threading
import time
import csv
from tqdm import tqdm
from datetime import datetime

############################################
# CONFIG
############################################

USER_AGENT = "SPA-Wikidata-reconciliation/3.0 (salgo60@msn.com)"

MAX_WORKERS = 12
RETRIES = 4
TIMEOUT = 20

QS_BATCH = 10000

TODAY = datetime.utcnow().strftime("%Y%m%d")

QS_PREFIX = f"spa_qs_{TODAY}"
AUDIT_FILE = f"spa_audit_{TODAY}.csv"
NETWORK_FILE = f"spa_network_{TODAY}.csv"
RESUME_FILE = f"spa_resume_{TODAY}.txt"

############################################
# SESSION
############################################

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

############################################
# CACHE
############################################

portrait_cache = {}
cache_lock = threading.Lock()

############################################
# RETRY FETCH
############################################

def fetch_portrait(spa_id):

    with cache_lock:
        if spa_id in portrait_cache:
            return portrait_cache[spa_id]

    url = f"https://portrattarkiv.se/endpoints/portraits.php?id={spa_id}"

    for attempt in range(RETRIES):

        try:

            r = session.get(url, timeout=TIMEOUT)

            if r.status_code != 200:
                raise Exception("HTTP error")

            data = r.json()
            
            p = data.get("persistent", {})
            
            facts = p.get("Facts", [])
            
            same = []
            
            for f in facts:
                if f.get("Type") == "SamePortrait":
                    same.append(f.get("Value"))
            
            portrait = {
                "id": spa_id,
                "first": p.get("FirstName"),
                "last": p.get("LastName"),
                "birth": p.get("BirthDate"),
                "same": same,
                "url": url
            }

            with cache_lock:
                portrait_cache[spa_id] = portrait

            return portrait

        except Exception:

            sleep = 2 ** attempt
            time.sleep(sleep)

    return None


############################################
# QUERY WIKIDATA
############################################

def get_wikidata_items():

    query = """
    SELECT ?item ?spa WHERE {
      ?item wdt:P4819 ?spa .
    }
    """

    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    sparql.addCustomHttpHeader("User-Agent", USER_AGENT)

    results = sparql.query().convert()

    items = {}

    for r in results["results"]["bindings"]:

        qid = r["item"]["value"].split("/")[-1]
        spa = r["spa"]["value"]

        items.setdefault(qid, set()).add(spa)

    return items


############################################
# LOAD RESUME
############################################

def load_resume():

    try:
        with open(RESUME_FILE) as f:
            return set(line.strip() for line in f)
    except:
        return set()


def save_resume(qid):

    with open(RESUME_FILE, "a") as f:
        f.write(qid + "\n")


############################################
# PROCESS ITEM
############################################

def process_item(record):

    qid, spa_ids = record

    qs_lines = []
    audit_rows = []
    network_rows = []

    for spa_id in spa_ids:

        source = fetch_portrait(spa_id)

        if not source:
            continue

        for target_id in source["same"]:

            network_rows.append((spa_id, target_id))
            network_rows.append((target_id, spa_id))

            if target_id in spa_ids:
                continue

            target = fetch_portrait(target_id)

            if not target:
                continue

            same_person = (
                source["first"] == target["first"]
                and source["last"] == target["last"]
                and source["birth"] == target["birth"]
            )

            audit_rows.append({
                "qid": qid,
                "source_id": spa_id,
                "target_id": target_id,
                "same_person": same_person,
                "first": source["first"],
                "last": source["last"],
                "birth": source["birth"]
            })

            if same_person:

                qs = (
                    f'{qid}\tP4819\t"{target_id}"\t'
                    f'S854\t"{target["url"]}"'
                )

                qs_lines.append(qs)

    return qs_lines, audit_rows, network_rows


############################################
# WRITE QS BATCHES
############################################

def write_qs_batches(qs_lines):

    qs_unique = sorted(set(qs_lines))

    batches = [
        qs_unique[i:i + QS_BATCH]
        for i in range(0, len(qs_unique), QS_BATCH)
    ]

    for i, batch in enumerate(batches, 1):

        filename = f"{QS_PREFIX}_{str(i).zfill(2)}.qs"

        with open(filename, "w", encoding="utf-8") as f:
            for line in batch:
                f.write(line + "\n")

        print("QS file written:", filename)


############################################
# MAIN
############################################

from tqdm import tqdm

def main():
    RESET_RUN = True
    items = get_wikidata_items()
    resume = set() if RESET_RUN else load_resume()
    print("Resume items:", len(resume))

    print("Total items:", len(items))

    qs_all = []
    audit_all = []
    network_all = []

    work_items = [
        record for record in items.items()
        if record[0] not in resume
    ]

    print("Remaining items:", len(work_items))

    with concurrent.futures.ThreadPoolExecutor(MAX_WORKERS) as executor:

        results = executor.map(process_item, work_items)

        for (qs, audit, network), record in tqdm(
            zip(results, work_items),
            total=len(work_items),
            desc="Processing SPA portraits",
            unit="items"
        ):

            qs_all.extend(qs)
            audit_all.extend(audit)
            network_all.extend(network)

            save_resume(record[0])

    ########################################
    # WRITE OUTPUT FILES
    ########################################

    write_qs_batches(qs_all)

    with open(AUDIT_FILE, "w", newline="", encoding="utf-8") as f:

        fieldnames = [
            "qid",
            "source_id",
            "target_id",
            "same_person",
            "first",
            "last",
            "birth"
        ]

        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in audit_all:
            writer.writerow(row)

    with open(NETWORK_FILE, "w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)
        writer.writerow(["source_spa_id", "target_spa_id"])

        for row in network_all:
            writer.writerow(row)
    same_links = sum(len(p["same"]) for p in portrait_cache.values())

    print("Portraits fetched:", len(portrait_cache))
    print("SamePortrait links:", same_links)

    print("\nFinished")
    print("QuickStatements:", len(qs_all))
    print("Audit rows:", len(audit_all))
    print("Network edges:", len(network_all))
    print("Portrait cache:", len(portrait_cache))
############################################

if __name__ == "__main__":
    main()

/var/folders/fd/md6r13sj0wsbg_6_xl160d300000gn/T/ipykernel_834/2264717093.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TODAY = datetime.utcnow().strftime("%Y%m%d")


Total items: 9429
Remaining items: 9429


Processing SPA portraits: 100%|██████████| 9429/9429 [11:13<00:00, 13.99items/s]

QS file written: spa_qs_20260312_01.qs
Portraits fetched: 23225
SamePortrait links: 34306

Finished
QuickStatements: 7749
Audit rows: 12481
Network edges: 27392
Portrait cache: 23225


In [3]:
p = fetch_portrait("sj9PGLAlnmUAAAAAABL8Cw")
print(p["same"])

['4e1scVvZ-EAAAAAAAAAD3Q', 'Dysd0OxEWfAAAAAAAABakQ', 'Dysd0OxEWfAAAAAAAABb7Q', 'sj9PGLAlnmUAAAAAABM1Vg', 'sj9PGLAlnmUAAAAAABNFhg']


In [5]:
import requests
import csv
from SPARQLWrapper import SPARQLWrapper, JSON
from tqdm import tqdm

USER_AGENT = "SPA-Wikidata-check/1.0 (salgo60@msn.com)"

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

QS_FILE = "spa_qs_20260312_01.qs"
REPORT = "spa_wd_validation.csv"  

def get_wd_person(qid):

    query = f"""
    SELECT ?label ?birth WHERE {{
      wd:{qid} rdfs:label ?label .
      FILTER(lang(?label)="sv" || lang(?label)="en")

      OPTIONAL {{ wd:{qid} wdt:P569 ?birth }}
    }}
    LIMIT 1
    """

    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    sparql.addCustomHttpHeader("User-Agent", USER_AGENT)

    r = sparql.query().convert()

    if not r["results"]["bindings"]:
        return None, None

    b = r["results"]["bindings"][0]

    label = b["label"]["value"]

    birth = None
    if "birth" in b:
        birth = b["birth"]["value"][:10]

    return label, birth

def get_spa_person(pid):

    url = f"https://portrattarkiv.se/endpoints/portraits.php?id={pid}"

    r = session.get(url)
    data = r.json()

    p = data["persistent"]

    name = f"{p.get('FirstName','')} {p.get('LastName','')}".strip()
    birth = p.get("BirthDate")

    return name, birth

rows = []

with open(QS_FILE) as f:
    lines = f.readlines()

for line in tqdm(lines):

    parts = line.split()

    qid = parts[0]
    pid = parts[2].replace('"','')

    wd_name, wd_birth = get_wd_person(qid)
    spa_name, spa_birth = get_spa_person(pid)

    status = "OK"

    if wd_birth and spa_birth and wd_birth != spa_birth:
        status = "BIRTH_DIFF"

    if wd_name and spa_name and spa_name.lower() not in wd_name.lower():
        status = "NAME_DIFF"

    rows.append({
        "qid": qid,
        "spa_id": pid,
        "wd_name": wd_name,
        "spa_name": spa_name,
        "wd_birth": wd_birth,
        "spa_birth": spa_birth,
        "status": status
    })

with open(REPORT, "w", newline="", encoding="utf8") as f:

    fieldnames = [
        "qid",
        "spa_id",
        "status",
        "wd_name",
        "spa_name",
        "wd_birth",
        "spa_birth"
    ]

    writer = csv.DictWriter(f, fieldnames=fieldnames)

    writer.writeheader()

    for r in rows:
        writer.writerow(r)

100%|███████████████████████████████████████| 7602/7602 [26:20<00:00,  4.81it/s]


In [7]:
import csv
import networkx as nx
from SPARQLWrapper import SPARQLWrapper, JSON
from datetime import datetime

NETWORK_FILE="spa_network_20260312.csv"
OUTPUT_HTML="spa_cluster_report.html"

USER_AGENT="SPA-cluster-analysis/1.0 (salgo60@msn.com)"

############################################
# Load SPA -> Wikidata mapping
############################################

print("Loading Wikidata mapping...")

query="""
SELECT ?item ?spa WHERE {
  ?item wdt:P4819 ?spa .
}
"""

sparql=SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
sparql.addCustomHttpHeader("User-Agent",USER_AGENT)

results=sparql.query().convert()

spa_to_qid={}

for r in results["results"]["bindings"]:

    qid=r["item"]["value"].split("/")[-1]
    spa=r["spa"]["value"]

    spa_to_qid[spa]=qid

print("Mappings:",len(spa_to_qid))

############################################
# Build SPA portrait graph
############################################

print("Loading SPA network...")

G=nx.Graph()

with open(NETWORK_FILE) as f:
    r=csv.DictReader(f)
    for row in r:
        G.add_edge(row["source_spa_id"],row["target_spa_id"])

print("Edges:",G.number_of_edges())

############################################
# Build clusters
############################################

clusters=list(nx.connected_components(G))

print("Clusters:",len(clusters))

############################################
# Analyse clusters
############################################

rows=[]

for cid,cluster in enumerate(clusters):

    qids=set()
    missing=[]

    for spa in cluster:

        if spa in spa_to_qid:
            qids.add(spa_to_qid[spa])
        else:
            missing.append(spa)

    status="OK"
    severity="green"

    if len(qids)>1:
        status="WD_DUPLICATE"
        severity="red"

    elif missing:
        status="MISSING_WD"
        severity="yellow"

    rows.append({
        "cluster_id":cid,
        "size":len(cluster),
        "status":status,
        "severity":severity,
        "qids":sorted(qids),
        "spa_ids":sorted(cluster)
    })

############################################
# Create HTML report
############################################

print("Generating HTML report...")

html=[]

html.append("""
<html>
<head>
<title>SPA Wikidata Cluster Report</title>
<style>

body {font-family: Arial}

table {border-collapse: collapse;width:100%}

th,td {
border:1px solid #ccc;
padding:6px;
font-size:13px;
}

tr.red {background:#ffcccc}
tr.yellow {background:#fff5b3}
tr.green {background:#e6ffe6}

a {text-decoration:none}

</style>
</head>
<body>
""")

html.append(f"<h2>SPA → Wikidata cluster analysis</h2>")
html.append(f"<p>Generated: {datetime.utcnow()}</p>")
html.append(f"<p>Total clusters: {len(rows)}</p>")

html.append("""
<table>
<tr>
<th>Cluster</th>
<th>Size</th>
<th>Status</th>
<th>Wikidata</th>
<th>SPA portraits</th>
</tr>
""")

for r in rows:

    qlinks=" ".join(
        f'<a href="https://www.wikidata.org/wiki/{q}" target="_blank">{q}</a>'
        for q in r["qids"]
    )

    splinks=" ".join(
        f'<a href="https://portrattarkiv.se/endpoints/portraits.php?id={p}" target="_blank">{p}</a>'
        for p in r["spa_ids"]
    )

    html.append(f"""
<tr class="{r['severity']}">
<td>{r['cluster_id']}</td>
<td>{r['size']}</td>
<td>{r['status']}</td>
<td>{qlinks}</td>
<td>{splinks}</td>
</tr>
""")

html.append("</table>")
html.append("</body></html>")

with open(OUTPUT_HTML,"w",encoding="utf8") as f:
    f.write("\n".join(html))

print("Report written:",OUTPUT_HTML)

Loading Wikidata mapping...
Mappings: 11131
Loading SPA network...
Edges: 13101
Clusters: 3910
Generating HTML report...
Report written: spa_cluster_report.html


/var/folders/fd/md6r13sj0wsbg_6_xl160d300000gn/T/ipykernel_834/3682293035.py:138: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  html.append(f"<p>Generated: {datetime.utcnow()}</p>")


In [9]:
import csv
import networkx as nx
from datetime import datetime
from SPARQLWrapper import SPARQLWrapper, JSON

NETWORK_FILE="spa_network_20260312.csv"
TODAY=datetime.utcnow().strftime("%Y%m%d")
OUTPUT_QS=f"SPA_add_P4819_{TODAY}.txt"

USER_AGENT="SPA-add-P4819/1.0 (salgo60@msn.com)"

############################################
# Load Wikidata SPA identifiers
############################################

query="""
SELECT ?item ?spa WHERE {
  ?item wdt:P4819 ?spa .
}
"""

sparql=SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
sparql.addCustomHttpHeader("User-Agent",USER_AGENT)

results=sparql.query().convert()

spa_to_qid={}
qid_to_spa={}

for r in results["results"]["bindings"]:

    qid=r["item"]["value"].split("/")[-1]
    spa=r["spa"]["value"]

    spa_to_qid[spa]=qid
    qid_to_spa.setdefault(qid,set()).add(spa)

############################################
# Build SPA portrait graph
############################################

G=nx.Graph()

with open(NETWORK_FILE) as f:
    r=csv.DictReader(f)

    for row in r:
        G.add_edge(row["source_spa_id"],row["target_spa_id"])

clusters=list(nx.connected_components(G))

############################################
# Create QuickStatements
############################################

qs_lines=[]

for cluster in clusters:

    qids=set()

    for spa in cluster:
        if spa in spa_to_qid:
            qids.add(spa_to_qid[spa])

    if len(qids)!=1:
        continue

    qid=list(qids)[0]

    existing=qid_to_spa[qid]

    for spa in cluster:

        if spa in existing:
            continue

        p=portrait_cache.get(spa)

        if not p:
            continue

        first=p.get("first")
        last=p.get("last")
        birth=p.get("birth")

        if not first or not last or not birth:
            continue

        qs=f'{qid}\tP4819\t"{spa}"'

        qs_lines.append(qs)

############################################
# Write file
############################################

with open(OUTPUT_QS,"w",encoding="utf8") as f:

    for line in sorted(set(qs_lines)):
        f.write(line+"\n")

print("New P4819 statements:",len(qs_lines))
print("File written:",OUTPUT_QS)

/var/folders/fd/md6r13sj0wsbg_6_xl160d300000gn/T/ipykernel_834/3432482511.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TODAY=datetime.utcnow().strftime("%Y%m%d")


New P4819 statements: 11974
File written: SPA_add_P4819_20260313.txt


In [10]:
from datetime import datetime
end = datetime.now()
print("Ended: ", end) 
print('Time elapsed (hh:mm:ss.ms) {}'.format(datetime.now() - start_time))

Ended:  2026-03-13 11:43:35.539433
Time elapsed (hh:mm:ss.ms) 17:18:01.661334
